In [ ]:
import sys
import os
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 确保 src/ 包可被导入
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, root_dir)

from config import COAL_TYPES, TRAIN_DIR, TEST_DIR, AUX_COLS, ALPHAS
from src.data   import load_labels, load_coal_spectra
from src.submit import pack_submission
from src.model import get_cv_splits
from src.features import compute_features
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler

In [2]:
label_map, aux_map = load_labels()
coal_type = COAL_TYPES[0]
train_data = load_coal_spectra(TRAIN_DIR, coal_type, label_map, aux_map)

In [22]:
def pool_coal_data(TRAIN_DIR, COAL_TYPES, label_map, aux_map):
    # Initialize the master dictionary
    pooled = {
        'spectra': [], 'stats': [], 'labs': [], 'lrel': [], 'rats': [],
        'targets': [], 'aux': [], 'groups': [],
        'names': [],
        'n_batches': 0,
        'coal_types': []  # New metadata array tracking the origin coal type
    }
    
    current_group_offset = 0
    expected_spectra_channels = None
    
    for coal_idx, coal_type in enumerate(COAL_TYPES):
        # Load data for the current coal type and compte features
        train_data = load_coal_spectra(TRAIN_DIR, coal_type, label_map, aux_map)
        n_shots = len(train_data['targets'])
        inorm = compute_features(train_data)
        
        # 1. Explicit Dimension Check for 'spectra'
        if expected_spectra_channels is None:
            expected_spectra_channels = inorm.shape[1]
        else:
            current_channels = inorm.shape[1]
            if current_channels != expected_spectra_channels:
                raise ValueError(
                    f"Dimension mismatch in 'spectra' for coal type '{coal_type}'. "
                    f"Expected {expected_spectra_channels} channels, got {current_channels}."
                )
                
        # 2. Handle 'groups' offset mapping
        # Add the current running total of batches to ensure coherent enumeration
        adjusted_groups = train_data['groups'] + current_group_offset
        current_group_offset += train_data['n_batches']
        
        # 3. Accumulate batch count
        pooled['n_batches'] += train_data['n_batches']
        
        # 4. Handle lists and metadata
        pooled['names'].extend(train_data['names'])
        pooled['coal_types'].extend([coal_idx] * n_shots)
        
        # 5. Temporarily store numpy arrays in lists for fast concatenation later
        pooled['spectra'].append(inorm)
        pooled['stats'].append(train_data['stats'])
        pooled['labs'].append(train_data['labs'])
        pooled['lrel'].append(train_data['lrel'])
        pooled['rats'].append(train_data['rats'])
        pooled['targets'].append(train_data['targets'])
        pooled['aux'].append(train_data['aux'])
        pooled['groups'].append(adjusted_groups)

    # 6. Perform a single, efficient concatenation for all numpy arrays
    array_keys = ['spectra', 'stats', 'labs', 'lrel', 'rats', 'targets', 'aux', 'groups']
    for key in array_keys:
        pooled[key] = np.concatenate(pooled[key], axis=0)
        
    # Convert metadata list to a numpy array for consistency with other shot-level data
    pooled['coal_types'] = np.array(pooled['coal_types'])
    
    return pooled

# --- Execution ---
# pooled_train_data = pool_coal_data(TRAIN_DIR, COAL_TYPES, label_map, aux_map)

In [23]:
pooled_train_data = pool_coal_data(TRAIN_DIR, COAL_TYPES, label_map, aux_map)

In [51]:
# PCA pooled globally
from config import N_PCA_MAX, RANDOM_STATE
from sklearn.decomposition import PCA
inorm_mat = pooled_train_data['spectra']
n_batches = pooled_train_data['n_batches']

scaler_spec = StandardScaler()
spec_scaled = scaler_spec.fit_transform(inorm_mat)
n_pca = min(N_PCA_MAX, n_batches - 1, inorm_mat.shape[0] - 1)
pca = PCA(n_components=n_pca, random_state=RANDOM_STATE)
spec_pca = pca.fit_transform(spec_scaled)

# Build feature matrix
hand_feats = np.hstack([pooled_train_data['stats'], pooled_train_data['labs'], 
                        pooled_train_data['lrel'], pooled_train_data['rats']])
X = np.hstack([spec_pca, hand_feats])
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
scaler_hand = StandardScaler()
X = scaler_hand.fit_transform(X)


In [52]:
import numpy as np

def build_pooled_design_matrix(X, coal_type_labels, coal_types):
    """
    X: (n_total_shots, p) pooled feature matrix (PCA + handcrafted, PCA fit globally)
    coal_type_labels: (n_total_shots,) array of coal type per row
    coal_types: list of the 5 coal type identifiers
    
    Returns: X_pooled (n, p + p*(n_types-1)) -- shared block + interaction blocks
             (one type is the reference/baseline, absorbed into the shared block)
    """
    n, p = X.shape
    n_coal_types = len(coal_types)
    dummies = np.stack([(coal_type_labels == ct).astype(float) for ct in range(1, n_coal_types)], axis=1)
    # shared block: X itself (applies to all rows, this is the "global" relationship)
    blocks = [X]
    # interaction blocks: X * dummy, one per non-reference coal type
    for j in range(dummies.shape[1]):
        blocks.append(X * dummies[:, [j]])
    X_pooled = np.hstack(blocks)  # shape (n, p * n_types)
    return X_pooled

In [53]:
coal_type_labels = pooled_train_data['coal_types']
design_matrix = build_pooled_design_matrix(X, coal_type_labels, COAL_TYPES)
design_matrix.shape

(944, 365)

In [55]:
def fit_hierarchical_ridge(X_pooled, y, n_types, alpha_shared, alpha_interaction):
    """
    alpha_shared: regularization on the shared/global block (light — trust pooled signal)
    alpha_interaction: regularization on type-specific deviation blocks (heavy — shrink toward global)
    """
    p = X_pooled.shape[1] // n_types
    y_mean = y.mean()
    y_centered = y - y_mean
    penalty_diag = np.concatenate([
        np.full(p, alpha_shared),
        np.full(p * (n_types - 1), alpha_interaction)
    ])
    D = np.diag(penalty_diag)
    XtX = X_pooled.T @ X_pooled
    XtY = X_pooled.T @ y_centered
    B = np.linalg.solve(XtX + D, XtY)
    return B, y_mean

def predict_hierarchical_ridge(X_pooled, B, y_mean):
    return X_pooled @ B + y_mean

In [56]:
def select_alphas_inner(X_tr, y_tr, batch_ids_tr, n_types,
                         alpha_shared_grid, alpha_interaction_grid, n_inner=3):
    inner_gkf = GroupKFold(n_splits=n_inner)
    best_rmse, best_params = np.inf, None
    for a_s in alpha_shared_grid:
        for a_i in alpha_interaction_grid:
            inner_oof = np.zeros(len(y_tr))
            for itr, ival in inner_gkf.split(X_tr, groups=batch_ids_tr):
                B, y_mean = fit_hierarchical_ridge(X_tr[itr], y_tr[itr], n_types, a_s, a_i)
                inner_oof[ival] = predict_hierarchical_ridge(X_tr[ival], B, y_mean)
            rmse = np.sqrt(np.mean((inner_oof - y_tr) ** 2))
            if rmse < best_rmse:
                best_rmse, best_params = rmse, (a_s, a_i)
    return best_params

In [57]:
from sklearn.model_selection import GroupKFold

groups = pooled_train_data['groups']
n_batches = pooled_train_data['n_batches']
aux = pooled_train_data['aux']
n_types = len(COAL_TYPES)

splits = get_cv_splits(groups, n_batches)
# X_pooled: pooled design matrix from build_pooled_design_matrix(X_spec, coal_type_labels, coal_types)
# p: number of columns in the ORIGINAL (non-expanded) feature matrix X_spec
# n_types: number of coal types (5)
# batch_id: batch identifier per row, aligned with X_pooled — needed for inner GroupKFold

alpha_shared_grid = [0.1, 1, 10, 100]
alpha_interaction_grid = [10, 100, 1000, 10000, 100000]  # wider range — expect these need to go high

predicted_aux_oof = np.zeros_like(aux, dtype=np.float32)
hier_models = {}
chosen_alphas_log = {}

for col_idx, col_name in enumerate(AUX_COLS):
    y_aux = aux[:, col_idx]

    if np.isnan(y_aux).any():
        predicted_aux_oof[:, col_idx] = float(np.nanmean(y_aux))
        hier_models[col_name] = None
        continue

    oof = np.zeros(len(y_aux))
    fold_alphas = []

    for tr_idx, val_idx in splits:  # your existing outer batch-grouped splits
        a_s, a_i = select_alphas_inner(
            design_matrix[tr_idx], y_aux[tr_idx], groups[tr_idx],
            n_types, alpha_shared_grid, alpha_interaction_grid
        )
        fold_alphas.append((a_s, a_i))
        B, y_mean = fit_hierarchical_ridge(design_matrix[tr_idx], y_aux[tr_idx], n_types, a_s, a_i)
        oof[val_idx] = predict_hierarchical_ridge(design_matrix[val_idx], B, y_mean)

    predicted_aux_oof[:, col_idx] = oof

    # Final refit on ALL data for inference — select alphas via same inner CV on full data
    a_s_final, a_i_final = select_alphas_inner(
        design_matrix, y_aux, groups, n_types, alpha_shared_grid, alpha_interaction_grid
    )
    B_final, y_mean_final = fit_hierarchical_ridge(design_matrix, y_aux, n_types, a_s_final, a_i_final)
    hier_models[col_name] = (B_final, y_mean_final, a_s_final, a_i_final)
    chosen_alphas_log[col_name] = {"per_fold": fold_alphas, "final": (a_s_final, a_i_final)}

    rmse = np.sqrt(np.mean((oof - y_aux) ** 2))
    print(f"Aux variable {col_name}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  Percentage error: {rmse / np.mean(y_aux) * 100:.2f}%")
    print(f"  alpha_shared/alpha_interaction per fold: {fold_alphas}")
    print(f"  Final (refit on all data): {a_s_final, a_i_final}")

Aux variable 全水分
  RMSE: 1.0393
  Percentage error: 11.32%
  alpha_shared/alpha_interaction per fold: [(100, 100), (10, 1000), (100, 100000), (100, 1000), (100, 100)]
  Final (refit on all data): (1, 100)
Aux variable 灰分
  RMSE: 4.8809
  Percentage error: 12.08%
  alpha_shared/alpha_interaction per fold: [(10, 100), (10, 10), (10, 10), (1, 10), (10, 1000)]
  Final (refit on all data): (10, 10)
Aux variable 氢
  RMSE: 0.1108
  Percentage error: 5.97%
  alpha_shared/alpha_interaction per fold: [(10, 100), (10, 10), (10, 10), (1, 10), (10, 1000)]
  Final (refit on all data): (10, 10)
Aux variable 硫
  RMSE: 0.0736
  Percentage error: 20.74%
  alpha_shared/alpha_interaction per fold: [(10, 100), (1, 1000), (1, 1000), (10, 10000), (1, 1000)]
  Final (refit on all data): (10, 100)


In [58]:
print(oof[:20])
print(y_aux[:20])
print("pred mean/std:", oof.mean(), oof.std())
print("true mean/std:", y_aux.mean(), y_aux.std())

[0.41807789 0.4085713  0.42867014 0.47181835 0.42635906 0.43519476
 0.44124747 0.4691212  0.44690184 0.41932718 0.42641204 0.38261015
 0.42863857 0.42598926 0.37713068 0.38776785 0.43299652 0.4550469
 0.35691366 0.44877564]
[0.56 0.56 0.56 0.56 0.56 0.56 0.56 0.56 0.56 0.56 0.56 0.56 0.51 0.51
 0.51 0.51 0.51 0.51 0.51 0.51]
pred mean/std: 0.3546531881212367 0.07224052134455769
true mean/std: 0.3546822033898305 0.09720623204140487


In [60]:
for col_idx, col_name in enumerate(AUX_COLS):
    print(f"--- {col_name} ---")
    for ct in range(len(COAL_TYPES)):
        mask = (coal_type_labels == ct) & ~np.isnan(aux[:, col_idx])
        rmse_ct = np.sqrt(np.mean((predicted_aux_oof[mask, col_idx] - aux[mask, col_idx]) ** 2))
        print(f"  {ct}: RMSE={rmse_ct:.4f}, n_batches={mask.sum()}")

--- 全水分 ---
  0: RMSE=1.3156, n_batches=140
  1: RMSE=0.6904, n_batches=86
  2: RMSE=1.8755, n_batches=96
  3: RMSE=0.5598, n_batches=262
  4: RMSE=0.9376, n_batches=360
--- 灰分 ---
  0: RMSE=7.1301, n_batches=140
  1: RMSE=4.1276, n_batches=86
  2: RMSE=6.1632, n_batches=96
  3: RMSE=3.8450, n_batches=262
  4: RMSE=4.2120, n_batches=360
--- 氢 ---
  0: RMSE=0.1604, n_batches=140
  1: RMSE=0.0973, n_batches=86
  2: RMSE=0.1380, n_batches=96
  3: RMSE=0.0891, n_batches=262
  4: RMSE=0.0952, n_batches=360
--- 硫 ---
  0: RMSE=0.0727, n_batches=140
  1: RMSE=0.0729, n_batches=86
  2: RMSE=0.0881, n_batches=96
  3: RMSE=0.0669, n_batches=262
  4: RMSE=0.0745, n_batches=360
